In [18]:
!pip install hmmlearn
import numpy as np
import hmmlearn as hmm
from hmmlearn import hmm

In [19]:
# a)

states = ["D", "M", "U"]
observations = ["FB", "B", "S", "NS"]

pi = np.array([1/3, 1/3, 1/3])
A = np.array([[0, 1/2, 1/2], # matricea de tranzitie
              [1/2, 1/4, 1/4],
              [1/2, 1/4, 1/4]])
B = np.array([[0.1, 0.2, 0.4, 0.3], # matricea de emisie
              [0.15, 0.25, 0.5, 0.1],
              [0.2, 0.3, 0.4, 0.1]])

print("suma pi:", pi.sum())
print("sume linii A:", A.sum(axis=1))
print("sume linii B:", B.sum(axis=1))

suma pi: 1.0
sume linii A: [1. 1. 1.]
sume linii B: [1. 1. 1.]


In [20]:
# b)

sequence = ["FB", "FB", "S", "B", "B", "S", "B", "B", "NS", "B", "B", "S"]
obs = [observations.index(nota) for nota in sequence]

T = len(obs)
N = len(states)

delta = np.zeros((T, N))
for j in range(N):
    delta[0][j] = pi[j] * B[j][obs[0]]

print("delta_1 =", delta[0])

delta_1 = [0.03333333 0.05       0.06666667]


In [21]:
# c)

psi = np.zeros((T, N), dtype=int)

for t in range(1, T):
    for j in range(N):
        scores = delta[t-1] * A[:, j]   # delta_{t-1}(i) * a_ij pentru fiecare i
        psi[t][j] = np.argmax(scores)
        delta[t][j] = B[j][obs[t]] * np.max(scores)

Explicatie rol $\psi_t(j)$: retine starea precedenta din care s-a obtinut scorul maxim $\delta_t(j)$, pentru a putea reconstitui traseul la final

In [22]:
# d)

print("delta_1 =", delta[0])
print("delta_2 =", delta[1])

# verificare manuala pentru delta_2(D)
# candidati: D->D, M->D, U->D
candidates = [delta[0][0] * A[0][0], delta[0][1] * A[1][0], delta[0][2] * A[2][0]]
print("candidati pentru D:", candidates)
print("delta_2(D) manual =", B[0][obs[1]] * max(candidates))

delta_1 = [0.03333333 0.05       0.06666667]
delta_2 = [0.00333333 0.0025     0.00333333]
candidati pentru D: [np.float64(0.0), np.float64(0.024999999999999998), np.float64(0.03333333333333333)]
delta_2(D) manual = 0.0033333333333333335


In [23]:
# e)

path = np.zeros(T, dtype=int)
path[T-1] = np.argmax(delta[T-1])

for t in range(T-2, -1, -1):
    path[t] = psi[t+1][path[t+1]]

best_prob = np.max(delta[T-1])

print("secventa de dificultati:", [states[i] for i in path])
print("max P(x, o) =", best_prob)

secventa de dificultati: ['D', 'U', 'D', 'U', 'D', 'M', 'D', 'U', 'D', 'U', 'D', 'M']
max P(x, o) = 2.1093750000000005e-11


f)

Viterbi cauta cea mai probabila secventa completa de dificultati, nu cea mai probabila dificultate separat la fiecare test:

$$\max_{x_1, \dots, x_{12}} P(x_1, \dots, x_{12}, o_1, \dots, o_{12})$$

Daca am alege independent la fiecare pas cea mai probabila stare, am putea obtine un traseu care nu respecta tranzitiile modelului (de exemplu $D \to D$, cu $a_{DD} = 0$). Viterbi tine cont de tranzitii si de emisii pe tot traseul.

Scorul de la punctul e) este o probabilitate comuna, nu una conditionata:

$$P(x^*, o) = \max_j \delta_{12}(j)$$

Valoarea este mica pentru ca include si probabilitatea de a observa exact acele 12 note. Ea nu reprezinta cat de sigur este traseul $x^*$ stiind notele. Pentru asta ar trebui impartita la $P(o)$, calculat cu algoritmul Forward:

$$P(x^* \mid o) = \frac{P(x^*, o)}{P(o)}$$

Deoarece $P(o)$ este acelasi pentru toate traseele, impartirea nu schimba care traseu este cel mai bun. De aceea Viterbi poate maximiza direct probabilitatea comuna.

In [24]:
# optional

alpha = np.zeros((T, N))
alpha[0] = pi * B[:, obs[0]]

for t in range(1, T):
    for j in range(N):
        alpha[t][j] = B[j][obs[t]] * np.sum(alpha[t-1] * A[:, j])

p_o = alpha[T-1].sum()
print("P(o) =", p_o)
print("P(x*|o) =", best_prob / p_o)


model = hmm.CategoricalHMM(n_components=3)
model.startprob_ = pi
model.transmat_ = A
model.emissionprob_ = B

X = np.array(obs).reshape(-1, 1)

log_p, lib_path = model.decode(X, algorithm="viterbi")
print("secventa biblioteca:", [states[i] for i in lib_path])
print("P(o) biblioteca =", np.exp(model.score(X)))
print("P(x*, o) biblioteca =", np.exp(log_p))

P(o) = 7.325942897172459e-08
P(x*|o) = 0.0002879322197302604
secventa biblioteca: ['U', 'U', 'D', 'U', 'D', 'M', 'D', 'U', 'D', 'U', 'D', 'M']
P(o) biblioteca = 7.32594289717248e-08
P(x*, o) biblioteca = 2.109375000000024e-11
